# Covered Today:
1. Selecting LLMs for Code Generation: Python to C++ with Cursor
2. Selecting Frontier Models: GPT, Claude, Grok & Gemini for C++ Code Gen
3. Porting Python to C++ with GPT
4. AI Coding Showdown: GPT vs Claude vs Gemini vs Groq Performance

Before moving ahead to topic 1, Ed spoke about the following:
1. As an LLM engineer, one of the most important task is to find the business problem we are trying to solve. When someone approaches us for a solution, we need to ask them what business problem are you trying to solve. It should be something tangible. The reason is that AI has received quite the hype and everyone wants a pie of that, however, AI may not solve for everything. 
2. Also, as an AI engineer, a person wears two hats: Software engineer and a Data Scientist. As per Ed, he gets most queries from learners related to SE. He says, these are not the most important questions. Most important questions are the Data Science parts: What is the business issue, how do we measure what are we solving for and what data do we have and what data do we need.

On the basis of above, he gives the below 5 steps to apply an LLM to a commercial problem(this will be covered later also.)
1. Understand: the business problem and how to measure it
2. Prepare: Select the best candidate models
3. Select: The model we are going to select
4. Customize: By building things like RAG or fine tuning the LLM
5. Production-ise: Scale

### 1. Selecting LLMs for Code Generation: Python to C++ with Cursor

Our next task is to write a program, which converts a given python script into an efficient C++ script. To do this, we start by viewing the benchmarks to shortlist the models we will be using for this task. Today we work with only frontier close sourced models and tomorrow we proceed with open source.

I will note down the steps taken next for identification of the candidate models:
1. We start with Artificial Analysis: we move to coding benchmarks. 

    First is Live code bench and the top 5 closed source models as per this benchmark are:
    1. Gemini 3 Pro Preview (high): 91.7 %
    2. Gemini 3 Flash Preview (Reasoning): 90.8%
    3. GPT 5.2 Medium: 89.4%
    4. GPT 5.2 xHigh: 88.9%
    5. Claude Opus 4.5: 87.1%. 

    Now, we move ahead with sci coding and the scores are:
    1. Claude Fable 5.1 (max with fallback): 63.1%
    2. Claude Fable 5 (with fallback): 61%
    3. Claude Fable 5.1 (xhigh with fallback): 60.9%
    4. Gemini 3.7 Flash (medium): 59.8%
    5. Gemini 3.7 Flash (medium): 59.7%

2. Now we move to vellum and in vellum we select coding LLMs, under this, we have two tests, and below are the scores
    LiveCodeBench
    1. Gemini 3 Pro: 79.7%
    2. Claude Opus 4.6: 76%
    3. OpenAI o3-mini: 74.1%
    4. Claude Sonnet 4.6: 72%
    5. GPT-4.1: 52%

    SWE Bench
    1. GPT-5.6 Sol: 96.2%
    2. Claude Mythos 5: 95.5%
    3. Claude Fable 5: 95%
    4. GPT-5.6 Luna: 93%
    5. Claude Opus 4.8: 88%

3. In the next step, we go to: SEAL and it has many benchmarks, below are the details:
    SWE Atlas - Refactoring: evaluates a model's ability to restructure production code while preserving behavior, which is our use case also.
    1. GPT 6 Astra (Codex) xHigh
    2. Fable-5.1 (Claude Code) xHigh
    3. Fable-5 (Claude Code)
    4. Opus-4.7 (Claude Code)
    5. Opus 4.8 (Claude Code)

4. Next we move to livebench, here under coding, we have the below leaders:
    1. Claude Fable 5.1 Max Effort
    2. Claude Fable 5 Max Effort
    3. GPT-5.6 Sol Max Effort
    4. GPT-5.2 Codex
    5. GPT-5.6 Luna Max Effort

5. And finally, we check Agent arena, here for coding, we have the below leaders:
    1. Claude Fable 5.1 (Max)
    2. GPT 6 Astra (Max)
    3. Claude Opus 5 (High)
    4. Claude Opus 5 (Max)
    5. Claude Fable 5 (High)


On the basis of above data, the final list of candidates is:
1. Claude Fable 5.1 (Max)
2. GPT 6 Astra (Codex) xHigh
3. Gemini 3 Pro

Adding gemini 3 pro, since I want to include models from major labs. In addition to these, gemini-3.1-pro-preview(latest pro version) and Sarvam-105B to the list to check how these hold up. 

In [1]:
# we now start by importing basic libraries, like openai, env, os to check if all models are working or not

In [2]:
from openai import OpenAI
from dotenv import load_dotenv
import os

In [3]:
load_dotenv(override=True)
openai_api = os.getenv("OPENAI_API_KEY")
google_api = os.getenv("GOOGLE_API_KEY")
claude_api = os.getenv("ANTHROPIC_API_KEY")
sarvam_api = os.getenv("SARVAM_API")

google_baseurl = "https://generativelanguage.googleapis.com/v1beta/openai/"
claude_baseurl = "https://api.anthropic.com/v1/"
sarvam_baseurl = "https://api.sarvam.ai/v1/"

In [4]:
# lets start by creating a function, that checks if APIs are working fine and models are responding or not.

def ping_llm(api_key, model, base_url):
  """Pings any OpenAI-compatible LLM endpoint.

  Returns True if reachable and responding, False otherwise.
  """
  if not api_key:
    print(f"[{model}] SKIPPED: API key is missing.")
    return False

  try:
    client = OpenAI(api_key=api_key, base_url=base_url)
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": "Respond only with: pong"}],
        temperature=0,
    )
    reply = response.choices[0].message.content.strip() #type: ignore
    print(f"[{model}] ONLINE -> {reply}")
    return "All APIs connected and working."
  except Exception as err:
    print(f"[{model}] FAILED -> {err}")
    return False

In [5]:
ping_llm(openai_api, "gpt-4o-mini", None)
ping_llm(google_api, "gemini-3.5-flash-lite", base_url=google_baseurl)
ping_llm(claude_api, "claude-haiku-4-5-20251001", base_url=claude_baseurl)
ping_llm(sarvam_api, "sarvam-105b", base_url=sarvam_baseurl)

[gpt-4o-mini] ONLINE -> pong
[gemini-3.5-flash-lite] ONLINE -> pong
[claude-haiku-4-5-20251001] ONLINE -> pong
[sarvam-105b] ONLINE -> pong


'All APIs connected and working.'

In [6]:
# Next is the system info that we extract from the system. This has been written wih an LLM and fetches the information from file mac_sysinfo.py. The code is an improvement over the course code, since, we have added caching information, and memory sizing info.
# Import the function from saved file
from mac_sysinfo import get_unified_m2_context
llm_system_context = get_unified_m2_context(as_json=True)
# Verify the output
print(llm_system_context)

{
  "os_environment": {
    "system": "Darwin",
    "release": "27.0.0",
    "architecture": "arm64",
    "rosetta2_translated": false,
    "target_triple": "arm64-apple-darwin27.0.0"
  },
  "hardware_constraints": {
    "cpu_brand": "Apple M2",
    "cores": {
      "physical": 8,
      "logical": 8
    },
    "compute_extensions": [
      "AdvSIMD",
      "AdvSIMD_HPFPCvt",
      "FEAT_BF16",
      "FEAT_DotProd",
      "FEAT_FHM",
      "FEAT_FP16",
      "FEAT_I8MM",
      "floatingpoint",
      "neon",
      "neon_fp16",
      "neon_hpfp"
    ],
    "memory": {
      "total_ram_bytes": 17179869184,
      "page_size_bytes": 16384
    },
    "cache": {
      "cache_line_size_bytes": 128,
      "l1_data_cache_bytes": 65536,
      "l2_cache_bytes": 4194304
    }
  },
  "toolchain": {
    "compilers": {
      "clang": "Apple clang version 21.0.0 (clang-2100.3.34.2)",
      "gcc": "Apple clang version 21.0.0 (clang-2100.3.34.2)"
    },
    "build_tools": {
      "make": "GNU Make 3.81",


In [7]:
# now since we have the system info with us, lets us move ahead with the python code as given in the course.

pi = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(200_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

In [8]:
# let's now run the code. Since, the code is in a str format, we use exec() to run this code.
exec(pi)

Result: 3.141592656089
Execution Time: 14.315763 seconds


In [ ]:
# Well, as we can see, the result is right, we wanted 3.14... as the output and the time taken by the machine is 14 seconds. Now, we have a python figure. Next, we move ahead to the c++ formatting. Now, since, I am not aware of C++ code, I will ask the LLMs to help me out in terms of how to compile the code and run it. 